In [ ]:
# Modules that need to be imported

import random
import string
import numpy as np

In [ ]:
# Read in the file that lists all the words that can be used as a Wordle guess
# Written as a function as it is used in the function game() later to define possible solution

def read_words(filename):
    with open(filename, "r") as f:
        return f.read().split("\n")

accepted_words = read_words("accepted_words.txt")

In [ ]:
# Function to simulate guessing a word outputting the colours that Wordle would show

# ans: the solution to the Wordle
# guess: the word that is being guessed

def word_score(ans, guess):
    return [
        "G" if g == a 
        else "Y" if g in ans 
        else "B"
        for g, a in zip(guess, ans)
    ]

In [ ]:
# Function to cut down the outstanding solution set after a guess has been made

# Sol_set: Remaining feasible solutions
# Word: Guess that has just been made
# Colours: Information recieved after the guess has been made

def update_sol_set(sol_set, word, colours):
    for i, (letter, col) in enumerate(list(zip(word, colours))):
        if col == "G":
            sol_set = [word for word in sol_set if word[i] == letter]
        if col == "Y":
            sol_set = [word for word in sol_set if letter in word and word[i] != letter]
        if col == "B":
            sol_set = [word for word in sol_set if letter not in word]
    return sol_set

# CODE ITERATES THROUGH SOL_SET MULTIPLE TIMES
# TO-DO: IMPROVE EFFICIENCY

In [ ]:
# Function to decide what the next guess in a game of Wordle should be

# sol_set: a list of the remaining possible answers to the puzzle
# used: a list of letters that have already been used in guesses
    # UNDERLYING IDEA: INFORMATION ABOUT NEW LETTERS IS PREFERABLE

def next_guess(sol_set,used):

    # For loop to count frequency with which letters appear across the solution set
    frequency = {letter:0 for letter in string.ascii_lowercase}
    for word in sol_set:
        for letter in word:
            frequency[letter] +=1

    # Only consider letters that occur somewhere in the solution set, and for which information is not already available

    frequency = {key: value for key, value in frequency.items() if key not in used and value != 0}
    ranked_letters = sorted(frequency, key=frequency.get, reverse=True)
    
    # The variable aim determines how many high frequency letters we want in our guess
    # Ideally there is a word which has 5 high frequency letters
    # If there isn't then we relax the condition

    for aim in range(5, 0, -1):
        for letters_required in range(5, 27 - len(used)):
            top_letters = ranked_letters[:letters_required]    #Choose the first letters_required most popular letters
            for guess in accepted_words:    
                if sum(letter in guess for letter in top_letters) == aim:           #Check if a word has enough of the popular letters
                    return guess

    # If no strong guess is found then just guess something
    return sol_set[0]

# COULD UNDERLYING ALGORITHM FOR SELECTING A GUESS BE IMPROVED


In [ ]:
# Function to simulate a game of Wordle

# ans: the solution to the game

def game(ans):
    possible_answers = read_words("possible_answers.txt")

    # First guess is always ROATE as per Starting_Word_Analysis.ipynb
    used = ["r","o","a","t","e"]
    cols = word_score(ans,"roate")                                          # Colours obtained from guessing ROATE
    possible_answers = update_sol_set(possible_answers, "roate", cols)      # Cut down solution set based on colours produced by ROATE  

    # Iterate rounds of Wordle
    num_guesses =1
    while len(possible_answers) >1:
        word = next_guess(possible_answers,used)                            # Determine next best guess
        used.extend(word)                                                   # Extend is like append but adds each element individually rather than appending the list itself
        cols = word_score(ans,word)                                         # Colours obtained by the guess
        possible_answers = update_sol_set(possible_answers, word, cols)     # Cut down solution set based on colours produced by guess
        num_guesses+=1
    return possible_answers[0], num_guesses+1